# Generate SLURM scripts for the pycistopic pipeline

This notebook reads template files and fills in user-specified paths/params to generate
self-contained bash scripts for each step of the pycistopic pipeline.

**Steps:**
1. `build/` — Build cisTopic object from h5ad
2. `cgs/` — Train LDA models (CGS)
3. `mallet/` — Train LDA models (MALLET, array job) — optional
4. `merge_mallet/` — Merge individual MALLET model files
5. `eval/` — Evaluate models

**Usage:**
1. Edit the configuration cell below with your paths and parameters
2. Run all cells — scripts are generated for **all subsets** at once
3. Generated `.sh` scripts will be written to `scripts_dir/{step}/{subset}_{step}.sh`
4. Submit each script via the SLURM wrapper (see submission instructions at the bottom)

In [1]:
import os

In [2]:
# ============================================================
# Configuration — edit this cell
# ============================================================

# Dataset name
dataset = "sc-islet-differentiation_10X-Multiome"

# Subsets to generate scripts for (one set of scripts per subset)
subsets = [
    "all",
    "all_replicate",
    "endocrine",
    "endocrine_replicate",
    "non-endocrine",
    "non-endocrine_replicate",
]

# Paths
venv_path = "/cellar/users/aklie/opt/deeptopic/.venv"
project_dir = "/cellar/users/aklie/opt/deeptopic"
templates_dir = f"{project_dir}/templates"
scripts_dir = f"/cellar/users/aklie/data/datasets/{dataset}/bin/4_topic_models/scripts"

# Base paths (subset name gets appended)
peak_calls_base = f"/cellar/users/aklie/data/datasets/{dataset}/results/2_process_data/peak_calls"
results_base = f"/cellar/users/aklie/data/datasets/{dataset}/results/4_topic_models"

# Shared input data
blacklist_path = f"/cellar/users/aklie/data/datasets/{dataset}/ref/blacklist.bed.gz"
cell_metadata_path = f"/cellar/users/aklie/data/datasets/{dataset}/results/2_process_data/subset/cell_metadata.tsv"

# Build params
region_separator = ":-"

# Training params
n_topics = "2 5 10 12 14 16 18 20 22 24 26 28 30 35 40 45 50"  # space-separated
n_iter = 150
n_cpu = 8
alpha = 50.0
eta = 0.1
seed = 555

# MALLET params (leave mallet_path empty to skip MALLET step)
mallet_path = "/cellar/users/aklie/opt/Mallet-202108/bin/mallet"

# Per-subset MALLET Java heap size.
subset_mallet_memory = {
    "endocrine_replicate":     "30G",
    "endocrine":               "40G",
    "non-endocrine_replicate": "60G",
    "non-endocrine":           "60G",
    "all_replicate":           "100G",
    "all":                     "120G",
}

# Per-subset SLURM resources
subset_slurm = {
    "endocrine_replicate":     {"mallet_mem": "48G",  "build_mem": "96G"},
    "endocrine":               {"mallet_mem": "64G",  "build_mem": "128G"},
    "non-endocrine_replicate": {"mallet_mem": "96G",  "build_mem": "128G"},
    "non-endocrine":           {"mallet_mem": "96G",  "build_mem": "192G"},
    "all_replicate":           {"mallet_mem": "128G", "build_mem": "256G"},
    "all":                     {"mallet_mem": "192G", "build_mem": "384G"},
}

# Evaluation params
groupby = "cell_type"

# ============================================================
# Phase 2: CREsted params (steps 5-6)
# ============================================================

# Binarization params (step 5)
binarize_method = "otsu"  # otsu, yen, li, aucell, ntop
binarize_n_topics = None  # None = auto-select best model, or set integer like 30

# CREsted training params (step 6)
genome_path = "/cellar/users/aklie/data/ref/genomes/hg38/hg38.fa"
crested_epochs = 100
crested_batch_size = 128
val_chroms = "chr8 chr10"
test_chroms = "chr9 chr18"

The history saving thread hit an unexpected error (DatabaseError('database disk image is malformed')).History will not be written to the database.


In [3]:
# ============================================================
# Derived paths — computed per subset
# ============================================================

n_topics_list = n_topics.split()
num_topics = len(n_topics_list)

# Build optional args
build_optional_args = []
if blacklist_path:
    build_optional_args.append(f"--blacklist_path {blacklist_path}")
if cell_metadata_path:
    build_optional_args.append(f"--cell_metadata_path {cell_metadata_path}")
if region_separator != ":-":
    build_optional_args.append(f"--region_separator {region_separator}")
build_optional_str = " \\\n    ".join(build_optional_args)

eval_optional_args = []
if groupby:
    eval_optional_args.append(f"--groupby {groupby}")
eval_optional_str = " \\\n    ".join(eval_optional_args)

# Binarize optional args
binarize_optional_args = []
if binarize_n_topics is not None:
    binarize_optional_args.append(f"--n_topics {binarize_n_topics}")
if binarize_method != "otsu":
    binarize_optional_args.append(f"--method {binarize_method}")
binarize_optional_str = " \\\n    ".join(binarize_optional_args)

# CREsted optional args
crested_optional_args = []
if crested_epochs != 100:
    crested_optional_args.append(f"--epochs {crested_epochs}")
if crested_batch_size != 128:
    crested_optional_args.append(f"--batch_size {crested_batch_size}")
if val_chroms != "chr8 chr10":
    crested_optional_args.append(f"--val_chroms {val_chroms}")
if test_chroms != "chr9 chr18":
    crested_optional_args.append(f"--test_chroms {test_chroms}")
crested_optional_str = " \\\n    ".join(crested_optional_args)

# Create step subdirectories
for step in ["build", "cgs", "mallet", "merge_mallet", "eval", "binarize", "train"]:
    os.makedirs(os.path.join(scripts_dir, step), exist_ok=True)

# Build per-subset config
subset_configs = {}
for subset in subsets:
    name = f"{dataset}_{subset}"
    output_base = os.path.join(results_base, subset)
    cfg = {
        "name": name,
        "subset": subset,
        "h5ad_path": os.path.join(peak_calls_base, subset, "peak_matrix.h5ad"),
        "build_output_dir": os.path.join(output_base, "objects"),
        "cgs_output_dir": os.path.join(output_base, "models_cgs"),
        "mallet_output_dir": os.path.join(output_base, "models_mallet"),
        "eval_output_dir": os.path.join(output_base, "evaluation"),
        "cistopic_obj_path": os.path.join(output_base, "objects", f"{name}.pkl"),
        "cgs_models_path": os.path.join(output_base, "models_cgs", "models.pkl"),
        "mallet_merged_path": os.path.join(output_base, "models_mallet", "merged"),
        # Phase 2 paths
        "binarize_output_dir": os.path.join(output_base, "binarized"),
        "beds_dir": os.path.join(output_base, "binarized", "beds"),
        "consensus_regions": os.path.join(output_base, "binarized", "consensus_regions.bed"),
        "crested_output_dir": os.path.join(output_base, "crested_model"),
    }
    subset_configs[subset] = cfg

print(f"Scripts dir: {scripts_dir}")
print(f"Topic range: {n_topics} ({num_topics} values)")
print(f"\nSubsets ({len(subsets)}):")
print(f"  {'Subset':30s}  {'MALLET --mem':>12s}  {'Java heap':>10s}  {'Build --mem':>11s}")
print(f"  {'-'*30}  {'-'*12}  {'-'*10}  {'-'*11}")
for s in subsets:
    slurm = subset_slurm[s]
    print(f"  {s:30s}  {slurm['mallet_mem']:>12s}  {subset_mallet_memory[s]:>10s}  {slurm['build_mem']:>11s}")

Scripts dir: /cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/bin/4_topic_models/scripts
Topic range: 2 5 10 12 14 16 18 20 22 24 26 28 30 35 40 45 50 (17 values)

Subsets (6):
  Subset                          MALLET --mem   Java heap  Build --mem
  ------------------------------  ------------  ----------  -----------
  all                                     192G        120G         384G
  all_replicate                           128G        100G         256G
  endocrine                                64G         40G         128G
  endocrine_replicate                      48G         30G          96G
  non-endocrine                            96G         60G         192G
  non-endocrine_replicate                  96G         60G         128G


## Generate all scripts

In [4]:
# Load templates
tmpl_build = open(os.path.join(templates_dir, "01_build_cistopic_obj.txt")).read()
tmpl_cgs = open(os.path.join(templates_dir, "02_run_models_cgs.txt")).read()
tmpl_mallet = open(os.path.join(templates_dir, "03_run_models_mallet.txt")).read()
tmpl_merge = open(os.path.join(templates_dir, "03b_merge_mallet_models.txt")).read()
tmpl_eval = open(os.path.join(templates_dir, "04_evaluate_models.txt")).read()
tmpl_binarize = open(os.path.join(templates_dir, "05_select_and_binarize.txt")).read()
tmpl_train = open(os.path.join(templates_dir, "06_train_crested.txt")).read()

def write_script(step, subset, content):
    """Write a script to scripts_dir/{step}/{subset}_{step}.sh"""
    path = os.path.join(scripts_dir, step, f"{subset}_{step}.sh")
    with open(path, "w") as f:
        f.write(content)
    return path

# Generate scripts for each subset
for subset, cfg in subset_configs.items():
    mallet_mem = subset_mallet_memory[subset]
    print(f"{'='*60}")
    print(f"Subset: {subset}  (mallet_memory={mallet_mem})")
    print(f"{'='*60}")

    # Build
    script = tmpl_build.format(
        venv_path, project_dir, cfg["h5ad_path"],
        cfg["build_output_dir"], cfg["name"], build_optional_str,
    )
    print(f"  Wrote: {write_script('build', subset, script)}")

    # CGS
    script = tmpl_cgs.format(
        venv_path, project_dir, cfg["cistopic_obj_path"],
        cfg["cgs_output_dir"], n_topics, n_cpu, n_iter, alpha, eta, seed,
    )
    print(f"  Wrote: {write_script('cgs', subset, script)}")

    # MALLET (array)
    if mallet_path:
        script = tmpl_mallet.format(
            venv_path, n_topics, project_dir, cfg["cistopic_obj_path"],
            cfg["mallet_output_dir"], n_cpu, n_iter, alpha, eta, seed,
            mallet_path, mallet_mem,
        )
        print(f"  Wrote: {write_script('mallet', subset, script)}")

        # Merge
        script = tmpl_merge.format(cfg["mallet_output_dir"])
        print(f"  Wrote: {write_script('merge_mallet', subset, script)}")

    # Eval
    script = tmpl_eval.format(
        venv_path, project_dir, cfg["cistopic_obj_path"],
        cfg["mallet_merged_path"], cfg["eval_output_dir"], eval_optional_str,
    )
    print(f"  Wrote: {write_script('eval', subset, script)}")

    # Binarize (step 5)
    script = tmpl_binarize.format(
        venv_path, project_dir, cfg["cistopic_obj_path"],
        cfg["mallet_merged_path"], cfg["binarize_output_dir"],
        binarize_optional_str,
    )
    print(f"  Wrote: {write_script('binarize', subset, script)}")

    # Train CREsted (step 6)
    script = tmpl_train.format(
        venv_path, project_dir, cfg["beds_dir"],
        cfg["consensus_regions"], genome_path,
        cfg["crested_output_dir"], cfg["name"],
        crested_optional_str,
    )
    print(f"  Wrote: {write_script('train', subset, script)}")

    print()

Subset: all  (mallet_memory=120G)
  Wrote: /cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/bin/4_topic_models/scripts/build/all_build.sh
  Wrote: /cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/bin/4_topic_models/scripts/cgs/all_cgs.sh
  Wrote: /cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/bin/4_topic_models/scripts/mallet/all_mallet.sh
  Wrote: /cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/bin/4_topic_models/scripts/merge_mallet/all_merge_mallet.sh
  Wrote: /cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/bin/4_topic_models/scripts/eval/all_eval.sh
  Wrote: /cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/bin/4_topic_models/scripts/binarize/all_binarize.sh
  Wrote: /cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/bin/4_topic_models/scripts/train/all_train.sh

Subset: all_replicate  (mallet_memory=100G)
  Wrote: /cellar

## Submission guide

### Resource budget
Target: **≤ 600G RAM, ≤ 100 CPUs** across all concurrent jobs.

### Per-subset resources (based on actual build usage)

| Subset | Build `--mem` | Build actual | MALLET `--mem` / `-c` | Eval `--mem` |
|--------|--------------|-------------|----------------------|-------------|
| endocrine_replicate | 96G | 53G | 48G / 8 | 32G |
| endocrine | 128G | 70G | 64G / 8 | 32G |
| non-endocrine_replicate | 128G | 82G | 96G / 8 | 32G |
| non-endocrine | 192G | 131G | 96G / 8 | 64G |
| all_replicate | 256G | 180G | 128G / 8 | 64G |
| all | 384G | 255G+ (OOM@256G) | 192G / 8 | 96G |

All jobs use `-t 14-00:00:00`. Merge uses `-m 4G -c 1`.

### Wave-based schedule
Run subsets in 3 waves to stay within the 600G budget while maximizing concurrency.

**Wave 1 — Small subsets**
| Subset | MALLET `--mem` | `-x` | Peak RAM |
|--------|---------------|------|----------|
| endocrine_replicate | 48G | 4 | 192G |
| endocrine | 64G | 2 | 128G |
| non-endocrine_replicate | 96G | 2 | 192G |
| **Total** | | **8 jobs** | **512G, 64 CPU** |

**Wave 2 — Medium subset**
| Subset | MALLET `--mem` | `-x` | Peak RAM |
|--------|---------------|------|----------|
| non-endocrine | 96G | 6 | 576G |
| **Total** | | **6 jobs** | **576G, 48 CPU** |

**Wave 3 — Large subsets (sequential)**
| Subset | MALLET `--mem` | `-x` | Peak RAM |
|--------|---------------|------|----------|
| all | 192G | 3 | 576G |
| all_replicate (after `all`) | 128G | 3 | 384G |
| **Total** | | **3 jobs** | **≤576G, 24 CPU** |

### Submission commands

```bash
SCRIPTS=scripts
LOGS=/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/bin/slurm_logs/4_topic_models
T=14-00:00:00
N=17  # number of topic values

# === BUILDS (submit all — fast, low CPU) ===
/cellar/users/aklie/opt/SLURM/cpu.sh -s $SCRIPTS/build/endocrine_replicate_build.sh     -j build_endocrine_replicate     -m 96G  -t $T -c 1 -o $LOGS/build
/cellar/users/aklie/opt/SLURM/cpu.sh -s $SCRIPTS/build/endocrine_build.sh               -j build_endocrine               -m 128G -t $T -c 1 -o $LOGS/build
/cellar/users/aklie/opt/SLURM/cpu.sh -s $SCRIPTS/build/non-endocrine_replicate_build.sh -j build_non-endocrine_replicate -m 128G -t $T -c 1 -o $LOGS/build
/cellar/users/aklie/opt/SLURM/cpu.sh -s $SCRIPTS/build/non-endocrine_build.sh           -j build_non-endocrine           -m 192G -t $T -c 1 -o $LOGS/build
/cellar/users/aklie/opt/SLURM/cpu.sh -s $SCRIPTS/build/all_replicate_build.sh           -j build_all_replicate           -m 256G -t $T -c 1 -o $LOGS/build
/cellar/users/aklie/opt/SLURM/cpu.sh -s $SCRIPTS/build/all_build.sh                     -j build_all                     -m 384G -t $T -c 1 -o $LOGS/build

# === WAVE 1: Small subsets (after their builds finish) ===
/cellar/users/aklie/opt/SLURM/cpu_array.sh -s $SCRIPTS/mallet/endocrine_replicate_mallet.sh     -j mallet_endocrine_replicate     -m 48G  -t $T -c 8 -n $N -x 8 -o $LOGS/mallet
/cellar/users/aklie/opt/SLURM/cpu_array.sh -s $SCRIPTS/mallet/endocrine_mallet.sh               -j mallet_endocrine               -m 64G  -t $T -c 8 -n $N -x 2 -o $LOGS/mallet
/cellar/users/aklie/opt/SLURM/cpu_array.sh -s $SCRIPTS/mallet/non-endocrine_replicate_mallet.sh -j mallet_non-endocrine_replicate -m 96G  -t $T -c 8 -n $N -x 2 -o $LOGS/mallet

# === WAVE 2: Medium subset (after wave 1 finishes) ===
/cellar/users/aklie/opt/SLURM/cpu_array.sh -s $SCRIPTS/mallet/non-endocrine_mallet.sh -j mallet_non-endocrine -m 96G -t $T -c 8 -n $N -x 4 -o $LOGS/mallet

# === WAVE 3: Large subsets (after wave 2) ===
/cellar/users/aklie/opt/SLURM/cpu_array.sh -s $SCRIPTS/mallet/all_mallet.sh           -j mallet_all           -m 192G -t $T -c 8 -n $N -x 3 -o $LOGS/mallet

# After all finishes:
/cellar/users/aklie/opt/SLURM/cpu_array.sh -s $SCRIPTS/mallet/all_replicate_mallet.sh -j mallet_all_replicate -m 128G -t $T -c 8 -n $N -x 3 -o $LOGS/mallet

# === MERGE + EVAL (after each subset's MALLET array completes) ===
# SUBSET=endocrine_replicate  # repeat for each subset
# /cellar/users/aklie/opt/SLURM/cpu.sh -s $SCRIPTS/merge_mallet/${SUBSET}_merge_mallet.sh -j merge_${SUBSET} -m 4G -t $T -c 1 -o $LOGS/merge_mallet
# /cellar/users/aklie/opt/SLURM/cpu.sh -s $SCRIPTS/eval/${SUBSET}_eval.sh -j eval_${SUBSET} -m 32G -t $T -c 1 -o $LOGS/eval
```

**Notes:**
- Each wave must wait for the previous wave to complete (to stay under the 600G budget)
- Within each wave, all subsets run concurrently
- Merge + eval can run as soon as a subset's MALLET array finishes
- Build mem values updated from actual sacct data (2026-03-02 runs)

## Phase 2: Binarize + Train CREsted (steps 5-6)

After evaluation, select a model, binarize topics to BED files, then train a DeepTopic CNN.

```bash
SCRIPTS=scripts
LOGS=/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/bin/slurm_logs/4_topic_models

# === STEP 5: Binarize topics (CPU, after eval) ===
SUBSET=endocrine_replicate
/cellar/users/aklie/opt/SLURM/cpu.sh \
  -s $SCRIPTS/binarize/${SUBSET}_binarize.sh \
  -j binarize_${SUBSET} \
  -c 1 \
  -m 32G \
  -t 01:00:00 \
  -o $LOGS/binarize

# === STEP 6: Train CREsted DeepTopic CNN (GPU, after binarize) ===
/cellar/users/aklie/opt/SLURM/gpu.sh \
  -s $SCRIPTS/train/${SUBSET}_train.sh \
  -j train_${SUBSET} \
  -p carter-gpu \
  -a carter-gpu \
  -g a30:1 \
  -c 4 \
  -m 32G \
  -t 08:00:00 \
  -o $LOGS/train
```

**Notes:**
- Step 5 auto-selects the best model unless `binarize_n_topics` is set in the config
- Step 6 requires a GPU node
- For the pilot run, use `endocrine_replicate` (smallest complete subset)

## Time and resource estimates per subset

In [5]:
import math

# Reference benchmark: endocrine_replicate (48k × 234k, 2.1 GB on disk)
# Observed: ~1.5h avg per MALLET job, ~2.7h max (n_topics=50), build ~9 min
ref_disk_gb = 2.1
ref_mallet_avg_h = 1.5  # average across topic values
ref_build_min = 9

print("=" * 90)
print(f"{'Subset':30s} {'Cells':>7s} {'Regions':>8s} {'Disk':>5s} {'Scale':>5s}  "
      f"{'Build':>6s} {'MALLET/job':>10s} {'MALLET total':>12s}")
print(f"{'':30s} {'':>7s} {'':>8s} {'(GB)':>5s} {'':>5s}  "
      f"{'(min)':>6s} {'(h, avg)':>10s} {'(job-h)':>12s}")
print("-" * 90)

total_job_hours = 0
for subset in subsets:
    cfg = subset_configs[subset]
    slurm = subset_slurm[subset]
    h5ad_path = cfg["h5ad_path"]
    disk_gb = os.path.getsize(h5ad_path) / 1e9
    scale = disk_gb / ref_disk_gb

    est_build_min = ref_build_min * scale
    est_mallet_avg_h = ref_mallet_avg_h * scale
    est_mallet_total_h = est_mallet_avg_h * num_topics

    # Read cell/region counts
    import anndata as ad
    adata = ad.read_h5ad(h5ad_path, backed="r")
    n_cells, n_regions = adata.n_obs, adata.n_vars
    adata.file.close()

    total_job_hours += est_mallet_total_h

    print(f"{subset:30s} {n_cells:>7,} {n_regions:>8,} {disk_gb:>5.1f} {scale:>5.1f}×  "
          f"{est_build_min:>6.0f} {est_mallet_avg_h:>10.1f} {est_mallet_total_h:>12.0f}")

print("-" * 90)
print(f"{'TOTAL':30s} {'':>7s} {'':>8s} {'':>5s} {'':>5s}  "
      f"{'':>6s} {'':>10s} {total_job_hours:>12.0f}")

# Wave estimates
wave1_h = max(
    math.ceil(num_topics / 4) * ref_mallet_avg_h * (os.path.getsize(subset_configs["endocrine_replicate"]["h5ad_path"]) / 1e9 / ref_disk_gb),
    math.ceil(num_topics / 2) * ref_mallet_avg_h * (os.path.getsize(subset_configs["endocrine"]["h5ad_path"]) / 1e9 / ref_disk_gb),
    math.ceil(num_topics / 2) * ref_mallet_avg_h * (os.path.getsize(subset_configs["non-endocrine_replicate"]["h5ad_path"]) / 1e9 / ref_disk_gb),
)
wave2_h = math.ceil(num_topics / 6) * ref_mallet_avg_h * (os.path.getsize(subset_configs["non-endocrine"]["h5ad_path"]) / 1e9 / ref_disk_gb)
wave3a_h = math.ceil(num_topics / 3) * ref_mallet_avg_h * (os.path.getsize(subset_configs["all"]["h5ad_path"]) / 1e9 / ref_disk_gb)
wave3b_h = math.ceil(num_topics / 3) * ref_mallet_avg_h * (os.path.getsize(subset_configs["all_replicate"]["h5ad_path"]) / 1e9 / ref_disk_gb)

print(f"\nWave schedule (600G RAM, 100 CPU budget):")
print(f"  Wave 1 (endo_rep ×4, endo ×2, non-endo_rep ×2):  ~{wave1_h:.0f}h")
print(f"  Wave 2 (non-endo ×6):                             ~{wave2_h:.0f}h")
print(f"  Wave 3a (all ×3):                                 ~{wave3a_h:.0f}h")
print(f"  Wave 3b (all_rep ×3):                             ~{wave3b_h:.0f}h")
print(f"  ────────────────────────────────────────────────────────")
print(f"  Estimated total wall time:                         ~{wave1_h + wave2_h + wave3a_h + wave3b_h:.0f}h ({(wave1_h + wave2_h + wave3a_h + wave3b_h)/24:.1f} days)")
print(f"  Total MALLET job-hours:                            ~{total_job_hours:.0f}h")

Subset                           Cells  Regions  Disk Scale   Build MALLET/job MALLET total
                                                 (GB)         (min)   (h, avg)      (job-h)
------------------------------------------------------------------------------------------
all                            104,080  605,517   6.9   3.3×      30        4.9           84
all_replicate                  104,080  382,312   6.1   2.9×      26        4.4           75
endocrine                       47,898  338,275   2.3   1.1×      10        1.6           28
endocrine_replicate             47,898  234,455   2.1   1.0×       9        1.5           26
non-endocrine                   56,182  516,983   4.2   2.0×      18        3.0           51
non-endocrine_replicate         56,182  300,795   3.7   1.7×      16        2.6           44
------------------------------------------------------------------------------------------
TOTAL                                                                       

# DONE!